# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR² mass spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `mlcroissant` library provides access to each logical table and its schema using the `dataset.metadata.record_sets` property.

For each record set, we print the `@id`, `name`, and an overview of its fields (columns by `@id`).

In [ ]:
# Explore the record sets present in the metadata
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No explicit record sets in metadata. Exploring primary files from dataset.distribution...")
    # Load one available default table (when recordSets is missing)
    default_record_set = dataset.metadata.default_record_set
    print(f"Default Record Set: @id={default_record_set.id}")
    print(f"  Name: {default_record_set.name}")
    print(f"  Description: {default_record_set.description}")
    print("  Fields / Columns:")
    for field in default_record_set.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    
    # Set up for extraction below
    record_sets = [default_record_set]
else:
    # Print info for each record set
    for rs in record_sets:
        print(f"Record Set: @id={rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        print("  Fields / Columns:")
        for field in rs.fields:
            print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we use the default (and only) record set.
# Reference all entities by their Croissant @id.

primary_record_set = record_sets[0]
primary_record_set_id = primary_record_set.id  # E.g. 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'

# List all available record sets by @id
record_set_ids = [rs.id for rs in record_sets]
# Store DataFrames for each record set
dataframes = {}

for rs in record_sets:
    print(f"Loading records for record set @id: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")

# Show the first few records
print(f"\nPreview of main dataframe (record set {primary_record_set_id}):")
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing numeric fields, and grouping.

**Note:** For this specific dataset, common useful fields might include measures like peptide counts, molecular weight, normalized abundance, etc. We'll identify and reference fields via their `@id` for full reproducibility.

In [ ]:
df = dataframes[primary_record_set_id]

# Inspect available columns/fields
print("Available columns/fields with their Croissant @id (column names):")
for c in df.columns:
    print("-", c)

# Choose numeric field by @id (Best guess: 'total_peptide_count' or similar)
# Let's try to auto-detect some possible numeric fields for demonstration
import re
numeric_candidates = [col for col in df.columns if re.search(r'count|abundance|MW|pI|coverage|mass', col, re.IGNORECASE)]
print("\nDetected possible numeric fields:", numeric_candidates)

# If present, let's use 'total_peptide_count' or fallback to first numeric match
numeric_field_id = None
for name in ['total_peptide_count', 'MW', 'molecular_weight', 'abundance', 'normalized_abundance', 'peptide_count']:
    if name in df.columns:
        numeric_field_id = name
        break
if not numeric_field_id and numeric_candidates:
    numeric_field_id = numeric_candidates[0]
if not numeric_field_id:
    raise Exception("No numeric field found for EDA.")

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Remove records with missing/non-numeric values in chosen field
df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")
print(filtered_df.head())

# Normalize the numeric field (z-score)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id]-filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Find a groupby/categorical field to summarize on
categorical_candidates = [col for col in df.columns if re.search(r'description|accession|sample|type|class|id', col, re.IGNORECASE)]
group_field_id = None
if categorical_candidates:
    group_field_id = categorical_candidates[0]
    print(f"\nGrouping by field: {group_field_id}")
    # Only group if not too many unique values
    if df[group_field_id].nunique() < 50:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id} (first five rows):")
        print(grouped_df.head())
else:
    print("No suitable categorical field found to group by.")

## 5. Visualization
Visualize the distribution of a key numeric field and (if available) the average by group.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if 'grouped_df' in locals():
    plt.figure(figsize=(12,6))
    order = grouped_df.sort_values(numeric_field_id, ascending=False)[group_field_id]
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df, order=order)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset with `mlcroissant`, explored its metadata, programmatically loaded and cleaned a primary data table by its Croissant `@id`, and performed basic exploratory and visual analytics. We filtered and normalized a key numeric field, and (optionally) explored means by group for categorical features. For more advanced use, repeat these steps with other datasets following this Croissant structure.

For reproducibility, all entities (record sets, fields, columns) were referenced by their Croissant `@id` field where applicable. For precise provenance or annotation, always refer back to the original Croissant schema and documentation at [https://sen.science/doi/10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
